<a href="https://colab.research.google.com/github/Savara-k/LLM-Fine-Tuning-and-RAG-Chatbot/blob/main/LLM_Fine_Tuning_and_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Overview**

This project demonstrates how to build and fine-tune a Large Language Model (LLM). The notebook walks through the core concepts behind modern NLP systems including prompt engineering, transformer models, parameter-efficient fine-tuning (LoRA), model evaluation using ROUGE metrics, and Retrieval Augmented Generation (RAG). The goal of the project is to create a knowledge-grounded chatbot capable of summarizing conversations and answering questions using retrieved information from a product knowledge base. This project was built as a hands-on exercise to better understand how LLM pipelines work in real-world AI applications.

#Project idea:
Build a knowledge-grounded e-commerce chatbot that includes:
* prompt engineering
* dialogue summarization
* LoRA fine-tuning
* ROUGE evaluation
* simple RAG with a vector store

---

#A good path is to use:
* FLAN-T5 for summarization/prompt engineering
* LoRA + PEFT for lightweight fine-tuning
* DialogSum for summarization data
* Sentence Transformers + ChromaDB for retrieval
* a simple Python chat loop instead of a full app first
* Hugging Face documents seq2seq tasks like summarization, PEFT/LoRA for  parameter-efficient fine-tuning, and the DialogSum dataset card describes the dialogue summarization dataset.
* Chroma’s docs describe it as an open-source retrieval engine for storing embeddings and querying similar text.

#Sections in the notebook:
1. Setup
2. Prompt engineering with FLAN-T5
3. Fine-tune with LoRA on DialogSum
4. Evaluate with ROUGE
5. Build a small RAG chatbot over product data
6. Demo examples
7. Talk with ChatBot
8. Final conclusions

# **1. Setup:**

In [1]:
!pip -q install transformers datasets evaluate rouge_score sentence-transformers chromadb peft accelerate bitsandbytes
!pip install -U transformers datasets evaluate peft accelerate

In [2]:
import os
import pandas as pd
import numpy as np
import torch

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    pipeline
)

from peft import LoraConfig, get_peft_model, TaskType
import evaluate

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


# **2. Prompt engineering with FLAN-T5:**

In [4]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [5]:
sample_dialogue = """
#Person1#: Hey, are we still meeting at 3?
#Person2#: Yes, but can we move it to 3:30?
#Person1#: Sure, that works for me.
#Person2#: Great, see you then.
"""

zero_shot_prompt = f"""
Summarize the following conversation in one sentence:

Conversation:
{sample_dialogue}

Summary:
"""

one_shot_prompt = f"""
Summarize the following conversation in one sentence.

Example:
Conversation:
#Person1#: I am hungry.
#Person2#: Let's order pizza.
Summary:
Two people decide to order pizza because one of them is hungry.

Now summarize this conversation:
Conversation:
{sample_dialogue}

Summary:
"""

inputs = tokenizer(zero_shot_prompt, return_tensors="pt", truncation=True).to(device)
outputs = base_model.generate(**inputs, max_new_tokens=50)
print("Zero-shot summary:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

inputs = tokenizer(one_shot_prompt, return_tensors="pt", truncation=True).to(device)
outputs = base_model.generate(**inputs, max_new_tokens=50)
print("\nOne-shot summary:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Zero-shot summary:
The meeting will be at 3 p.m.

One-shot summary:
The meeting will be moved to 3:30.


# Load dataset for fine-tuning

In [6]:
dataset = load_dataset("knkarthick/dialogsum")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [7]:
dataset["train"][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

In [8]:
small_train = dataset["train"].shuffle(seed=42).select(range(500))
small_test = dataset["test"].shuffle(seed=42).select(range(100))

print("Train size:", len(small_train))
print("Test size:", len(small_test))

Train size: 500
Test size: 100


# Tokenize data:

In [9]:
max_input_length = 512
max_target_length = 64

def preprocess_function(examples):

    inputs = [f"Summarize this dialogue:\n{dialogue}" for dialogue in examples["dialogue"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    labels_ids = labels["input_ids"]

    # IMPORTANT: Replace padding tokens with -100
    labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels_ids
    ]

    model_inputs["labels"] = labels_ids

    return model_inputs

In [10]:
tokenized_train = small_train.map(preprocess_function, batched=True, remove_columns=small_train.column_names)
tokenized_test = small_test.map(preprocess_function, batched=True, remove_columns=small_test.column_names)

# **3. Fine-tune with LoRA on DialogSum:**

In [11]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q", "v"]
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


# **4. Evaluate with ROUGE**

In [12]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Convert predictions from logits to token IDs (re-applying the fix)
    predictions = np.argmax(predictions, axis=-1)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}

# **5. Train the model:**

In [13]:
training_args = TrainingArguments(
    output_dir="./llm_lora_summarizer",
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    fp16=False,
    report_to="none",
    disable_tqdm=True
)

In [14]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=peft_model)

In [15]:
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

train_results = trainer.train()

{'loss': '1.797', 'grad_norm': '0.5562', 'learning_rate': '9.24e-05', 'epoch': '0.16'}
{'loss': '1.761', 'grad_norm': '0.5292', 'learning_rate': '8.44e-05', 'epoch': '0.32'}
{'loss': '1.654', 'grad_norm': '0.4676', 'learning_rate': '7.64e-05', 'epoch': '0.48'}
{'loss': '1.517', 'grad_norm': '0.5377', 'learning_rate': '6.84e-05', 'epoch': '0.64'}
{'loss': '1.571', 'grad_norm': '0.7054', 'learning_rate': '6.04e-05', 'epoch': '0.8'}
{'loss': '1.586', 'grad_norm': '0.5862', 'learning_rate': '5.24e-05', 'epoch': '0.96'}
{'eval_loss': '1.53', 'eval_rouge1': '0.3423', 'eval_rouge2': '0.1387', 'eval_rougeL': '0.3058', 'eval_rougeLsum': '0.3065', 'eval_runtime': '1.05', 'eval_samples_per_second': '95.24', 'eval_steps_per_second': '23.81', 'epoch': '1'}
{'loss': '1.45', 'grad_norm': '0.5897', 'learning_rate': '4.44e-05', 'epoch': '1.12'}
{'loss': '1.445', 'grad_norm': '0.4257', 'learning_rate': '3.64e-05', 'epoch': '1.28'}
{'loss': '1.436', 'grad_norm': '0.6184', 'learning_rate': '2.84e-05', 'ep

In [16]:
sample_batch = {
    "input_ids": torch.tensor([tokenized_train[0]["input_ids"]]).to(device),
    "attention_mask": torch.tensor([tokenized_train[0]["attention_mask"]]).to(device),
    "labels": torch.tensor([tokenized_train[0]["labels"]]).to(device),
}

peft_model.train()
outputs = peft_model(**sample_batch)
print("Manual loss:", outputs.loss.item())

Manual loss: 1.620503306388855


In [17]:
trainer.train()

{'loss': '1.348', 'grad_norm': '0.5659', 'learning_rate': '9.24e-05', 'epoch': '0.16'}
{'loss': '1.448', 'grad_norm': '0.6986', 'learning_rate': '8.44e-05', 'epoch': '0.32'}
{'loss': '1.464', 'grad_norm': '0.512', 'learning_rate': '7.64e-05', 'epoch': '0.48'}
{'loss': '1.365', 'grad_norm': '0.6856', 'learning_rate': '6.84e-05', 'epoch': '0.64'}
{'loss': '1.411', 'grad_norm': '0.7532', 'learning_rate': '6.04e-05', 'epoch': '0.8'}
{'loss': '1.439', 'grad_norm': '0.6285', 'learning_rate': '5.24e-05', 'epoch': '0.96'}
{'eval_loss': '1.477', 'eval_rouge1': '0.4084', 'eval_rouge2': '0.1744', 'eval_rougeL': '0.3772', 'eval_rougeLsum': '0.3769', 'eval_runtime': '1.047', 'eval_samples_per_second': '95.53', 'eval_steps_per_second': '23.88', 'epoch': '1'}
{'loss': '1.366', 'grad_norm': '0.6512', 'learning_rate': '4.44e-05', 'epoch': '1.12'}
{'loss': '1.355', 'grad_norm': '0.5064', 'learning_rate': '3.64e-05', 'epoch': '1.28'}
{'loss': '1.346', 'grad_norm': '0.7202', 'learning_rate': '2.84e-05', '

TrainOutput(global_step=250, training_loss=1.3917201614379884, metrics={'train_runtime': 19.2196, 'train_samples_per_second': 52.03, 'train_steps_per_second': 13.008, 'train_loss': 1.3917201614379884, 'epoch': 2.0})

In [18]:
print(tokenized_train[0]["labels"][:30])

[9637, 65, 29, 31, 17, 718, 7588, 21, 3, 9, 307, 97, 5, 216, 3088, 160, 12, 817, 160, 3, 88, 530, 3, 9, 5546, 11, 3, 88, 4227, 207]


In [19]:
# Re-instantiate the Trainer to ensure a clean state for evaluation.
# This addresses the 'on_train_begin must be called before on_evaluate' error.
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Call train to properly initialize internal states for evaluation callbacks.
trainer.train()

# Now, perform the evaluation
eval_results = trainer.evaluate()
eval_results
trainer.evaluate()

{'loss': '1.28', 'grad_norm': '0.6083', 'learning_rate': '9.24e-05', 'epoch': '0.16'}
{'loss': '1.383', 'grad_norm': '0.6567', 'learning_rate': '8.44e-05', 'epoch': '0.32'}
{'loss': '1.419', 'grad_norm': '0.6153', 'learning_rate': '7.64e-05', 'epoch': '0.48'}
{'loss': '1.304', 'grad_norm': '0.7022', 'learning_rate': '6.84e-05', 'epoch': '0.64'}
{'loss': '1.377', 'grad_norm': '0.9138', 'learning_rate': '6.04e-05', 'epoch': '0.8'}
{'loss': '1.422', 'grad_norm': '0.7444', 'learning_rate': '5.24e-05', 'epoch': '0.96'}
{'eval_loss': '1.456', 'eval_rouge1': '0.4214', 'eval_rouge2': '0.1853', 'eval_rougeL': '0.3919', 'eval_rougeLsum': '0.3916', 'eval_runtime': '1.05', 'eval_samples_per_second': '95.25', 'eval_steps_per_second': '23.81', 'epoch': '1'}
{'loss': '1.316', 'grad_norm': '0.6564', 'learning_rate': '4.44e-05', 'epoch': '1.12'}
{'loss': '1.322', 'grad_norm': '0.5455', 'learning_rate': '3.64e-05', 'epoch': '1.28'}
{'loss': '1.304', 'grad_norm': '0.7524', 'learning_rate': '2.84e-05', 'e

{'eval_loss': 1.4541244506835938,
 'eval_rouge1': 0.4247,
 'eval_rouge2': 0.1855,
 'eval_rougeL': 0.3955,
 'eval_rougeLsum': 0.3948,
 'eval_runtime': 1.0477,
 'eval_samples_per_second': 95.443,
 'eval_steps_per_second': 23.861,
 'epoch': 2.0}

# Compare summaries before and after fine-tuning

In [20]:
def generate_summary(model, dialogue, max_new_tokens=50):
    prompt = f"Summarize this dialogue:\n{dialogue}"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [21]:
sample = small_test[0]

print("DIALOGUE:\n")
print(sample["dialogue"])

print("\nREFERENCE SUMMARY:\n")
print(sample["summary"])

print("\nMODEL SUMMARY:\n")
print(generate_summary(peft_model, sample["dialogue"]))

DIALOGUE:

#Person1#: Hi! What are you watching?
#Person2#: It's a program about islam. It's very interesting.
#Person1#: Wow! So many people! Where are they and what are they doing?
#Person2#: They are muslims on a pilgrimage to mecca. Muslims call this pilgrimage 'haj'.
#Person1#: Why do they go there?
#Person2#: Muslims believe that every man who is able should go on a haj at least once in his life. Mecca is the spiritual centre of the muslim faith.
#Person1#: When muslims pray, they face towards mecca.
#Person2#: That's right. Unfortunately, so many people go on the haj each year that there are often stamped and people get killed.
#Person1#: I heard about that. The pilgrims must walk around a large, sacred black stone.
#Person2#: That's right. That's when accidents often happen. The Saudi government tries to limit the number of pilgrims, to reduce the chances of accidents.
#Person1#: Pilgrimages are common in many faiths.
#Person2#: Yes. In England, Christian pilgrims might go to C

# **6. Build a simple RAG chatbot for e-commerce:**

In [22]:
product_docs = [
    {
        "id": "1",
        "text": "Product: Running Shoes. Price: $89. Category: Footwear. Features: Lightweight, breathable mesh, ideal for daily running.",
        "data": {"category": "footwear"}
    },
    {
        "id": "2",
        "text": "Product: Wireless Headphones. Price: $129. Category: Electronics. Features: Noise cancellation, 30-hour battery life, Bluetooth 5.3.",
        "data": {"category": "electronics"}
    },
    {
        "id": "3",
        "text": "Product: Coffee Maker. Price: $59. Category: Kitchen. Features: 12-cup capacity, programmable timer, auto shut-off.",
        "data": {"category": "kitchen"}
    },
    {
        "id": "4",
        "text": "Product: Laptop Backpack. Price: $49. Category: Accessories. Features: Water-resistant, fits 15-inch laptop, multiple compartments.",
        "data": {"category": "accessories"}
    },
]

In [23]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
collection = client.create_collection(name="products")

for doc in product_docs:
    embedding = embedding_model.encode(doc["text"]).tolist()
    collection.add(
        ids=[doc["id"]],
        documents=[doc["text"]],
        metadatas=[doc["metadata"]],
        embeddings=[embedding]
    )

print("Knowledge base created.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base created.


In [24]:
def retrieve_docs(query, top_k=2):
    query_embedding = embedding_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    return results["documents"][0]

In [25]:
def rag_chatbot(query, model):
    retrieved = retrieve_docs(query, top_k=2)
    context = "\n".join(retrieved)

    prompt = f"""
You are a helpful e-commerce assistant.
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_new_tokens=80)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "query": query,
        "retrieved_context": retrieved,
        "answer": answer
    }

# **7. Demo examples:**

In [26]:
response = rag_chatbot("Which product is good for commuting with a laptop in the rain?", peft_model)

print("QUESTION:")
print(response["query"])

print("\nRETRIEVED CONTEXT:")
for item in response["retrieved_context"]:
    print("-", item)

print("\nANSWER:")
print(response["answer"])

QUESTION:
Which product is good for commuting with a laptop in the rain?

RETRIEVED CONTEXT:
- Product: Laptop Backpack. Price: $49. Category: Accessories. Features: Water-resistant, fits 15-inch laptop, multiple compartments.
- Product: Wireless Headphones. Price: $129. Category: Electronics. Features: Noise cancellation, 30-hour battery life, Bluetooth 5.3.

ANSWER:
Laptop Backpack


# **8. Talk with ChatBot:**

In [27]:
while True:
    user_query = input("Ask about products (or type 'quit'): ")
    if user_query.lower() == "quit":
        break

    response = rag_chatbot(user_query, peft_model)
    print("\nBot:", response["answer"])
    print("\nRetrieved:")
    for item in response["retrieved_context"]:
        print("-", item)
    print("\n" + "="*60 + "\n")

Ask about products (or type 'quit'): what is Laptop Backpack price?

Bot: $49.

Retrieved:
- Product: Laptop Backpack. Price: $49. Category: Accessories. Features: Water-resistant, fits 15-inch laptop, multiple compartments.
- Product: Wireless Headphones. Price: $129. Category: Electronics. Features: Noise cancellation, 30-hour battery life, Bluetooth 5.3.


Ask about products (or type 'quit'): how much are wireless headphones ?

Bot: $129.

Retrieved:
- Product: Wireless Headphones. Price: $129. Category: Electronics. Features: Noise cancellation, 30-hour battery life, Bluetooth 5.3.
- Product: Running Shoes. Price: $89. Category: Footwear. Features: Lightweight, breathable mesh, ideal for daily running.


Ask about products (or type 'quit'): what category is Running shoes?

Bot: Footwear

Retrieved:
- Product: Running Shoes. Price: $89. Category: Footwear. Features: Lightweight, breathable mesh, ideal for daily running.
- Product: Laptop Backpack. Price: $49. Category: Accessories. 

# **Final conclusions:**

This project demonstrates the full workflow of building a modern NLP application using Large Language Models. Starting from foundational concepts such as prompt engineering and dialogue summarization, the project progresses into more advanced techniques including parameter-efficient fine-tuning with LoRA and Retrieval Augmented Generation (RAG).

Through this process, a FLAN-T5 model was fine-tuned on the DialogSum dataset to improve its ability to summarize conversations. The model was evaluated using ROUGE metrics, showing measurable improvements in summarization performance after fine-tuning.

In addition to fine-tuning, the project implemented a simple RAG pipeline using sentence embeddings and a vector database. This allowed the chatbot to retrieve relevant information from a small product knowledge base before generating responses, helping ensure that answers are grounded in real data rather than relying solely on the model’s internal knowledge.

#Overall, the project highlights several key lessons:

 * Pretrained language models provide strong general capabilities but benefit from domain-specific fine-tuning.
 * Parameter Efficient Fine-Tuning (PEFT) techniques like LoRA allow models to be adapted efficiently without retraining the entire network.
 * Prompt engineering plays an important role in shaping model outputs.
 * Retrieval Augmented Generation helps reduce hallucinations by grounding responses in external information sources.

This project provides a practical foundation for building real-world LLM systems and can be extended further by adding larger datasets, reinforcement learning with human feedback (RLHF), or deploying the chatbot as a web application.